# Worked review: 2D Hessian ingredients from exact node means
### The user's attempt, corrected answers, executable Python, and readiness questions

**Scope:** Sections 1–3 only. We are validating a known quadratic, exact uniform regional means, and split-gradient records. **No fitted CART, real data, least squares, or Hessian reconstruction yet.**

**Research provenance:** Prepared November 16, 2026 for a review of work associated with **Week 46 (November 9–15, 2026)**. This is a retrospective worked-review artifact, **not evidence the completed notebook existed or was executed during Week 46**.

**Numerical example reused from the previous \`quadratic_hessian.ipynb\`:**
\[
c=0.7,\qquad b=\begin{pmatrix}-1\\2\end{pmatrix},\qquad
H_{\rm true}=\begin{pmatrix}4&1.5\\1.5&3\end{pmatrix}.
\]

This file provides **solutions** because you explicitly requested them. The final questions remain unanswered so you can demonstrate independent understanding to Nathan. The earlier notebook used **point-evaluation finite differences**, not exact regional means; those are different experiments.

## 1. Review of your answers: known quadratic

\[
f(x_1,x_2)=c+b^\top x+\frac12x^\top H_{\rm true}x,
\quad x=\begin{pmatrix}x_1\\x_2\end{pmatrix}.
\]

| Question | Assessment | Correction or confirmation |
|---|---|---|
| **1.1 Dimensions** | **Partially correct** | \(c\) is a scalar (a \(1\times1\) value), \(b\) and \(x\) are each \(2\times1\) column vectors, \(b^\top\) is \(1\times2\), and \(H\) is \(2\times2\). You called \(b\) a \(1\times2\) row; that's the dimension of \(b^\top\), not \(b\) in this notation. |
| **1.2 Interaction** | **Correct** | \(H_{12}=H_{21}=1.5\) are the mixed-curvature entries. Together with the factor \(1/2\), they contribute \(H_{12}x_1x_2\) to \(f\). |
| **1.3 \(f(0,0)\)** | **Correct** | Both the linear and quadratic terms vanish; \(f(0,0)=c=0.7\). |
| **1.4 Python implementation** | **Incorrect as written** | Python requires explicit multiplication and a vector. \`*\` on NumPy arrays is elementwise, while \`@\` is matrix multiplication. \`1/2 (x1,x2)\` is not valid Python. Use \`x = np.array([x1,x2])\`, \`b @ x\`, and \`x @ H_true @ x\`. |

**Key distinction:** A correct mathematical expression is not automatically valid NumPy syntax. The next cell is the complete, readable implementation.

In [ ]:
import numpy as np

# Reuse the numerical example from quadratic_hessian.ipynb.
c = 0.7
b = np.array([-1.0, 2.0])
H_true = np.array([[4.0, 1.5],
                   [1.5, 3.0]])

def f(x1, x2):
    """Evaluate f(x) = c + b^T x + 1/2 x^T H_true x."""
    x = np.array([x1, x2], dtype=float)
    return float(c + b @ x + 0.5 * (x @ H_true @ x))

assert H_true.shape == (2, 2)
np.testing.assert_allclose(H_true, H_true.T)
np.testing.assert_allclose(f(0, 0), 0.7)
print("f(0,0) =", f(0, 0))
print("f(0.5,0.5) =", f(0.5, 0.5))

## 2. Review of your answers: exact uniform box means

The rectangular node is
\[
B=[l_1,u_1]\times[l_2,u_2].
\]
Its center is the **2-vector** \(m_B=((l_1+u_1)/2,(l_2+u_2)/2)\), and its widths are \(w_1=u_1-l_1\) and \(w_2=u_2-l_2\).

| Question | Assessment | Correction or confirmation |
|---|---|---|
| **2.1 Center coordinates** | **Correct** | Your two midpoint expressions are right. Do **not** average them together: a 2D point needs both coordinates. |
| **2.2 Box widths** | **Partially correct** | \(w_1=u_1-l_1\) and \(w_2=u_2-l_2\) are correct. \(u_2-l_1\) is **never a width of this axis-aligned rectangle**, even if the numbers happen to be close: it subtracts endpoints of different coordinates. |
| **2.3 Why mean differs from center value** | **Incorrect** | It is because \(f\) is **curved**, not because of the split axis. Averaging a quadratic over a region generally differs from evaluating it at the center. |
| **2.4 Meaning of correction** | **Incorrect** | The correction captures the contribution of curvature along the two coordinate directions. **It does not require an even split.** Under a uniform rectangular average, the centered \(x_1x_2\) cross-product averages to zero, so no *additional* \(H_{12}\) term is needed; its contribution at the center is already in \(f(m_B)\). |
| **2.5 Unit box comparison** | **Incorrect** | Here \(f(0.5,0.5)=2.45\) and the exact mean is **2.7416667**, which is **larger** because \(H_{11},H_{22}>0\). |
| **2.6 Your box_mean code** | **Incorrect** | Your expression computes a **scalar average of two midpoint coordinates**, not an average of function values. You must evaluate \`f\` at the 2D center and then add the curvature correction. |

For a uniform distribution in a rectangle, \(\operatorname{E}(X_i-m_i)^2=w_i^2/12\). Since the quadratic's second-order term is multiplied by \(1/2\), its expected extra contribution is \(H_{ii}w_i^2/24\). Thus

\[
\boxed{\operatorname{mean}_B f=f(m_1,m_2)+\frac{H_{11}w_1^2+H_{22}w_2^2}{24}.}
\]

This is an **exact expression under uniform averaging over the box**. It is not automatically the empirical mean of responses observed in a CART node.

In [ ]:
def box_mean(l1, u1, l2, u2):
    """Exact uniform average of the quadratic f on a 2D rectangle."""
    if not (l1 < u1 and l2 < u2):
        raise ValueError("Each coordinate interval must have positive width.")

    center_x1 = (l1 + u1) / 2.0
    center_x2 = (l2 + u2) / 2.0
    width_x1 = u1 - l1
    width_x2 = u2 - l2

    center_value = f(center_x1, center_x2)
    curvature_correction = (
        H_true[0, 0] * width_x1**2 +
        H_true[1, 1] * width_x2**2
    ) / 24.0

    return center_value + curvature_correction

print("Unit-box center:", (0.5, 0.5))
print("Function at center:", f(0.5, 0.5))
print("Exact unit-box mean:", box_mean(0, 1, 0, 1))
print("Curvature correction:", box_mean(0, 1, 0, 1) - f(0.5, 0.5))
np.testing.assert_allclose(box_mean(0, 1, 0, 1), 2.7416666666666667, atol=1e-12)

# Shrinking the box reduces the curvature correction.
tiny = box_mean(0.49, 0.51, 0.49, 0.51)
print("Tiny box mean:", tiny)
print("Difference from center value:", tiny - f(0.5, 0.5))

## 3. Review of your answers: gradient contrast and effective location

**Two child boxes must retain both coordinates.**

- For \`axis=0\` (split on \(x_1\) at \(\tau\)):  
  lower \([l_1,\tau]\times[l_2,u_2]\); upper \([\tau,u_1]\times[l_2,u_2]\).
- For \`axis=1\` (split on \(x_2\) at \(\tau\)):  
  lower \([l_1,u_1]\times[l_2,\tau]\); upper \([l_1,u_1]\times[\tau,u_2]\).

For split axis \(s\), calculate both means with \`box_mean\` and return
\[
\gamma_s=\frac{\mu_{\rm upper}-\mu_{\rm lower}}{(u_s-l_s)/2}.
\]
The **effective point** has two coordinates:
\[
m_s=\frac{l_s+\tau+u_s}{3},\qquad
m_k=\frac{l_k+u_k}{2}\quad(k\ne s).
\]

| Question | Assessment | Correction or confirmation |
|---|---|---|
| **3.1 axis=0 children** | **Incorrect/incomplete** | You omitted the second coordinate interval and used \`u2\` where \`u1\` was needed. Use the full rectangles above. |
| **3.2 axis=1 children** | **Incorrect** | \(\tau\) absolutely matters: now it divides the \(x_2\) interval, while \(x_1\) remains unchanged. |
| **3.3 Effective location** | **Incorrect** | A scalar expression is insufficient. Unit square, \`axis=0\`, \(\tau=0.4\): \(m=((0+0.4+1)/3,(0+1)/2)=(0.4666667,0.5)\). |
| **3.4 Why half the parent width** | **Incorrect reasoning** | The lower child center along \(s\) is \((l_s+\tau)/2\), and the upper center is \((\tau+u_s)/2\). Their separation is **exactly** \((u_s-l_s)/2\), even for unequal splits. |
| **3.5 Function implementation** | **Incorrect** | \`box_mean\` is called **twice** using valid lower- and upper-child 2D bounds. The denominator is the **parent width along the chosen split axis divided by two**, not \`u2-l1\`. |
| **3.6 Predict gamma** | **Incomplete** | You gave a procedure rather than a numerical prediction. For the specified example, lower mean \(=2.2566667\), upper mean \(=3.065\), and \(\gamma_1=1.6166667\). |

**Important:** The estimated gradient component \(\gamma_s\) is a scalar. The effective location \(m\) is a vector. They are **not** the same kind of object.

In [ ]:
def gradient_estimate(l1, u1, l2, u2, tau, axis):
    """Return (gamma, m) from exact uniform child means.

    axis=0: slope-like contrast in x1.
    axis=1: slope-like contrast in x2.
    m: 2-vector where gamma equals the corresponding derivative for
       the quadratic with exact uniform box means.
    """
    if axis not in (0, 1):
        raise ValueError("axis must be 0 (x1) or 1 (x2).")
    if not (l1 < u1 and l2 < u2):
        raise ValueError("The parent box must have positive widths.")

    if axis == 0:
        if not (l1 < tau < u1):
            raise ValueError("For axis=0, require l1 < tau < u1.")
        mean_lower = box_mean(l1, tau, l2, u2)
        mean_upper = box_mean(tau, u1, l2, u2)
        parent_width = u1 - l1
        m = np.array([(l1 + tau + u1) / 3.0,
                      (l2 + u2) / 2.0])
    else:
        if not (l2 < tau < u2):
            raise ValueError("For axis=1, require l2 < tau < u2.")
        mean_lower = box_mean(l1, u1, l2, tau)
        mean_upper = box_mean(l1, u1, tau, u2)
        parent_width = u2 - l2
        m = np.array([(l1 + u1) / 2.0,
                      (l2 + tau + u2) / 3.0])

    gamma = (mean_upper - mean_lower) / (parent_width / 2.0)
    return float(gamma), m

## 4. Numerical trace: show every input and result

**Example 1:** Parent \([0,1]\times[0,1]\), split on \(x_1\) at \(0.4\).

- Lower child: \([0,0.4]\times[0,1]\), mean \(2.2566667\)
- Upper child: \([0.4,1]\times[0,1]\), mean \(3.065\)
- Parent width along \(x_1\): \(1\)
- Slope \(\gamma_1=(3.065-2.2566667)/0.5=1.6166667\)
- Effective point \(m=(0.4666667,0.5)\).

**Example 2:** Parent \([0,1]\times[0,1]\), split on \(x_2\) at \(0.4\).

- Lower child: \([0,1]\times[0,0.4]\), mean \(1.4966667\)
- Upper child: \([0,1]\times[0.4,1]\), mean \(3.5716667\)
- Slope \(\gamma_2=4.15\)
- Effective point \(m=(0.5,0.4666667)\).

For this quadratic, \(\nabla f(m)=b+H_{\rm true}m\). **We use that expression only as an independent check:** the estimator itself uses child-node means and box geometry, not an evaluation of the true gradient.

In [ ]:
for axis in (0, 1):
    print("\\n--- Split axis", axis, "---")
    tau = 0.4
    if axis == 0:
        mu_lower = box_mean(0, tau, 0, 1)
        mu_upper = box_mean(tau, 1, 0, 1)
    else:
        mu_lower = box_mean(0, 1, 0, tau)
        mu_upper = box_mean(0, 1, tau, 1)
    gamma, m = gradient_estimate(0, 1, 0, 1, tau, axis)
    true_gradient_at_m = b + H_true @ m
    print("Lower mean:", mu_lower)
    print("Upper mean:", mu_upper)
    print("Gamma:", gamma)
    print("Effective point m:", m)
    print("True derivative at m:", true_gradient_at_m[axis])
    np.testing.assert_allclose(gamma, true_gradient_at_m[axis], atol=1e-12)

np.testing.assert_allclose(
    gradient_estimate(0, 1, 0, 1, 0.4, axis=0)[0], 1.6166666666666667,
    atol=1e-12
)
np.testing.assert_allclose(
    gradient_estimate(0, 1, 0, 1, 0.4, axis=1)[0], 4.15,
    atol=1e-12
)
print("\\nBoth axis checks passed.")

## 5. Geometric picture: why \`axis\` changes the child boxes

This cell visualizes the same **unequal split at \(\tau=0.4\)** for \`axis=0\` and \`axis=1\`. The black dot is the effective location of the corresponding slope estimate, not the location of an observation in the dataset.

In [ ]:
# Optional visualization (requires matplotlib)
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for axis, ax in enumerate(axes):
    ax.add_patch(Rectangle((0, 0), 1, 1, fill=False, linewidth=2))
    if axis == 0:
        ax.axvline(0.4, linestyle="--", color="black")
        ax.text(0.2, 0.8, "lower child", ha="center")
        ax.text(0.7, 0.8, "upper child", ha="center")
    else:
        ax.axhline(0.4, linestyle="--", color="black")
        ax.text(0.5, 0.2, "lower child", ha="center")
        ax.text(0.5, 0.7, "upper child", ha="center")
    gamma, m = gradient_estimate(0, 1, 0, 1, 0.4, axis)
    ax.scatter([m[0]], [m[1]], s=70, color="black", zorder=3)
    ax.annotate(f"m=({m[0]:.3f},{m[1]:.3f})",
                xy=m, xytext=(0.05, 0.05),
                arrowprops={"arrowstyle": "->"})
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect("equal")
    ax.set(xlabel="$x_1$", ylabel="$x_2$", title=f"axis={axis}, gamma={gamma:.4f}")
plt.show()

## 6. A deliberate incorrect effective location

A plausible bug is replacing the split-axis effective coordinate \((l_s+\tau+u_s)/3\) with the parent midpoint \((l_s+u_s)/2\). **This does not change the child means or the slope \(\gamma\)**; it changes only the point where the slope is claimed to estimate a derivative.

For \`axis=0\`, \(\tau=0.4\), the correct effective point is \((0.4666667,0.5)\), whereas the incorrect midpoint is \((0.5,0.5)\). Compare the exact gradient component at these two locations. The midpoint check should fail because the partial derivative changes with \(x_1\).

This is a **location-validation failure**, not yet a failed reconstructed Hessian entry. We will test Hessian reconstruction explicitly only after you can defend Sections 1–3.

In [ ]:
gamma, m_correct = gradient_estimate(0, 1, 0, 1, 0.4, axis=0)
m_wrong = np.array([0.5, 0.5])
derivative_correct = (b + H_true @ m_correct)[0]
derivative_wrong = (b + H_true @ m_wrong)[0]

print("Same gamma:", gamma)
print("Correct m:", m_correct, "true derivative:", derivative_correct)
print("Wrong m:", m_wrong, "true derivative:", derivative_wrong)
print("Mismatch when using wrong location:", gamma - derivative_wrong)

np.testing.assert_allclose(gamma, derivative_correct, atol=1e-12)
assert not np.isclose(gamma, derivative_wrong)
print("Deliberately wrong location correctly fails the derivative check.")

## 7. Independent numerical sanity checks

A single hand-picked example is useful but cannot establish general correctness. These checks sample **different rectangular bounds and unequal split thresholds** while keeping the same quadratic. For each, the split-derived gradient should match the analytical derivative at its effective location.

This is a test of the code against a known mathematical identity, **not** evidence that real-data CART node means satisfy the exact uniform-mean assumptions.

In [ ]:
rng = np.random.default_rng(20261116)
max_error = 0.0
for _ in range(100):
    lower = rng.uniform(-2, 1, size=2)
    widths = rng.uniform(0.2, 2, size=2)
    upper = lower + widths
    axis = int(rng.integers(0, 2))
    frac = float(rng.uniform(0.05, 0.95))
    tau = lower[axis] + frac * widths[axis]
    gamma, m = gradient_estimate(
        lower[0], upper[0], lower[1], upper[1], tau, axis
    )
    expected = (b + H_true @ m)[axis]
    max_error = max(max_error, abs(gamma - expected))
    np.testing.assert_allclose(gamma, expected, rtol=1e-11, atol=1e-11)
print("100 random geometry checks passed.")
print("Maximum absolute discrepancy:", max_error)

# Fail early on invalid box/split specifications.
for args in [(0, 1, 0, 1, 0, 0), (0, 1, 0, 1, 1, 1), (1, 1, 0, 1, 0.5, 0)]:
    try:
        gradient_estimate(*args)
        raise AssertionError(f"Expected ValueError for {args}")
    except ValueError:
        pass
print("Invalid-input checks passed.")

## 8. Questions to answer independently **before** building \(H_{11},H_{12},H_{22}\)

Answer each in your own words **without consulting the solution above**. The aim is to know what the running code means, not to memorize its output.

1. **Dimensions:** If \(b\) is a 2-vector, why does \`b @ x\` produce a scalar? What would \`b * x\` produce?  
   **My answer:** ...

2. **Box vs number:** Is the center of a 2D box a scalar or a point? Write its coordinates for \([0.2,0.8]\times[-1,3]\).  
   **My answer:** ...

3. **Mean vs center value:** If \(H_{11}=H_{22}=0\), how would \`box_mean\` compare with \`f\` at the center? What happens to the correction?  
   **My answer:** ...

4. **Mixed term:** Why can \(H_{12}\ne0\) even though the box-mean width correction contains only \(H_{11}\) and \(H_{22}\)?  
   **My answer:** ...

5. **Uneven split:** For parent \([0,1]\times[0,1]\), \`axis=0\`, \(\tau=0.2\), write **both** child rectangles and the distance between their centers *along \(x_1\)*.  
   **My answer:** ...

6. **Axis change:** In \`axis=1\`, which coordinate bounds get replaced by \(\tau\) in the lower and upper calls to \`box_mean\`?  
   **My answer:** ...

7. **Effective point:** For parent \([0,1]\times[0,2]\), \`axis=1\`, \(\tau=0.5\), calculate **both** coordinates of \(m\).  
   **My answer:** ...

8. **Why 1/3:** What incorrect assumption are you making if you evaluate the split contrast at the parent midpoint instead of its effective point? Explain using the \(\tau=0.4\) example.  
   **My answer:** ...

9. **Trace the program:** What are the **inputs, outputs, and assumptions** of \`f\`, \`box_mean\`, and \`gradient_estimate\`? Which function calls which?  
   **My answer:** ...

10. **Verification:** What independent calculation confirms that \(\gamma_1=1.6166667\) at \((0.4666667,0.5)\)? Why is matching a single value insufficient for general verification?  
    **My answer:** ...

11. **Transfer to CART:** When you replace exact uniform means with observed child sample means, which mathematical assumption is no longer guaranteed, even if the code runs?  
    **My answer:** ...

12. **Research ownership:** If your future reconstructed Hessian is symmetric only after \((H+H^\top)/2\), what diagnostic may that averaging conceal?  
    **My answer:** ...

**Stopping rule:** Be able to answer 1–10 and reproduce one axis-0 and one axis-1 trace without help before starting explicit 2D Hessian reconstruction. Questions 11–12 prepare you for the later real-data stage.

## 9. Meeting-ready evidence and limitations

**What this notebook actually establishes (conditional on successful execution):**
- The quadratic is implemented with the previously chosen numerical coefficients.
- Exact uniform node means can be calculated from the rectangular bounds and the quadratic.
- Two exact child means yield a directional derivative contrast.
- The claimed effective location matches the true derivative for the quadratic in axis-0, axis-1, and additional test boxes.
- An incorrect midpoint location can be made to fail a known-derivative check.

**What it does not establish:**
- No \(H_{11},H_{12},H_{22}\) has been reconstructed in this notebook.
- No fitted regression tree was used.
- No empirical node means were used.
- No OPFData or IEEE-118 curvature was estimated.
- Passing numerical checks for this quadratic is not a general theorem for arbitrary functions.

### My own 60-second explanation to Nathan

**What I implemented:** [write in my own words]

**What I verified numerically:** [write in my own words]

**My earlier misconception:** [write in my own words]

**What I still cannot justify:** [write in my own words]

**Next bounded experiment:** Reconstruct \(H_{11}\), \(H_{12}\), \(H_{22}\) from explicitly chosen split patterns using only these functions and compare to \(H_{\rm true}\). Do not invoke CART or least squares yet.